<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03b_feature_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_feature_application**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [3]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [4]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

### 0.3. Definición de rutas



In [6]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [7]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [8]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.4. Códigos auxiliares para carga de datos y visualización


In [9]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [10]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

# **1. Carga de dataset**

In [11]:
mnq_intraday = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (894845, 21)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


# **2. Indicadores Técnicos**

## **2.1. Indicadores técnicos individuales**

In [12]:
import pandas as pd
from typing import List, Tuple, Iterable
from ta.volatility import AverageTrueRange
from ta.momentum import ROCIndicator

def calcular_indicadores_tecnicos(
    df: pd.DataFrame,
    *,
    target: str = "close",
    date_col: str = "date",
    momentum_windows: Tuple[int, int] = (10, 5),
    ema_span: int = 60,
    atr_windows: Iterable[int] = (14, 20),
    roc_windows: Tuple[int, int, int] = (20, 30, 60),
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Calcula indicadores técnicos por día (groupby(date_col)):
      - Momentum: pct_change(w)
      - EMA normalizada: close / EMA(span) - 1
      - ATR normalizado: ATR(w) / close
      - ROC: ROC(w)

    Retorna:
      (df_con_indicadores, indicator_columns)
    """
    # -------------------------
    # Columnas a crear
    # -------------------------
    mom_cols = [f"mom_{w}" for w in momentum_windows]
    ema_cols = [f"ema_{ema_span}"]
    atr_cols = [f"atr_norm_{w}" for w in atr_windows]
    roc_cols = [f"roc_{w}" for w in roc_windows]

    indicator_columns = mom_cols + ema_cols + atr_cols + roc_cols

    # -------------------------
    # Cálculo por día
    # -------------------------
    def aplicar_por_dia(grupo: pd.DataFrame) -> pd.DataFrame:
        grupo = grupo.copy()

        # ATR normalizado
        for w in atr_windows:
            atr = AverageTrueRange(
                high=grupo["high"],
                low=grupo["low"],
                close=grupo[target],
                window=int(w),
            )
            grupo[f"atr_norm_{w}"] = atr.average_true_range() / grupo[target]

        # EMA normalizada
        grupo[f"ema_{ema_span}"] = grupo[target] / grupo[target].ewm(span=ema_span).mean() - 1

        # Momentum
        for w in momentum_windows:
            grupo[f"mom_{w}"] = grupo[target].pct_change(w)

        # ROC
        for w in roc_windows:
            grupo[f"roc_{w}"] = ROCIndicator(close=grupo[target], window=int(w)).roc()

        return grupo

    out = df.groupby(date_col, group_keys=False).apply(aplicar_por_dia)
    return out, indicator_columns


## **2.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [13]:
import pandas as pd

# ============================================================
# Calcular indicadores
# ============================================================

mnq_intraday, indicator_columns = calcular_indicadores_tecnicos(
    mnq_intraday,
    target="close",
    date_col="date",
    momentum_windows=(10, 5),
    ema_span=60,
    atr_windows=(14, 20),
    roc_windows=(20, 30, 60),
)

# Asegurar índice datetime (si aplica)
mnq_intraday.index = pd.to_datetime(mnq_intraday.index)

In [14]:
info_mnq_indicators = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_indicators)

Dataset: mnq_intraday
Shape: (894845, 29)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


Filtramos todos los NaNs del dataset

In [15]:
# Eliminación de filas con NaN
mnq_intraday = mnq_intraday.dropna()

In [16]:
info_mnq_indicators = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_indicators)

Dataset: mnq_intraday
Shape: (700595, 29)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [17]:
assert not mnq_intraday.isna().any().any(), \
    "El dataset contiene NaN"

# **3. Introducción de interacciones**

## **3.1. Implementación**

In [18]:
def add_interactions(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    # Indicadores base (se asume que ya existen en df)
    ema_col: str = "ema_60",
    roc60_col: str = "roc_60",
    roc20_col: str = "roc_20",
    roc30_col: str = "roc_30",
    mom5_col: str = "mom_5",
    mom10_col: str = "mom_10",
    atr14_col: str = "atr_norm_14",
    atr20_col: str = "atr_norm_20",
    # Nombres de salida
    prefix: str = "",
    # Opcional: controlar si crear atr-interactions
    add_atr_interactions: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Crea interacciones mínimas, interpretables y compatibles con tu set final:
      - Full day: ema_60, roc_60 (+ atr_norm_20 en H=90)
      - Gestation: ema_60, roc_60, roc_20, roc_30, mom_10, mom_5
      - Execution: ema_60, roc_60, roc_20, roc_30, atr_norm_14

    Importante:
      - Calcula interacciones para TODO el dataset (toda la jornada).
      - Luego se evalúan IC/ventana filtrando por horario en tu pipeline.
      - Lags y slope se calculan por día (groupby(date).shift(1)) => sin leakage.

    Devuelve:
      - df_out: dataframe con nuevas columnas
      - created_cols: lista con los nombres creados
    """
    df_out = df.copy()

    # -------------------------
    # Helper para nombres
    # -------------------------
    def _col(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    # -------------------------
    # Validaciones mínimas (solo lo imprescindible)
    # -------------------------
    needed = [date_col, ema_col, roc60_col]
    missing = [c for c in needed if c not in df_out.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas para interacciones: {missing}")

    created_cols: list[str] = []

    # ATR x ROC (contexto de volatilidad)
    # - atr_norm_20: más coherente para full_day H=90
    # - atr_norm_14: más coherente para execution
    if add_atr_interactions:
        if atr20_col in df_out.columns:
            roc60_x_atr20 = _col("roc60_x_atr20")
            df_out[roc60_x_atr20] = df_out[roc60_col] * df_out[atr20_col]
            created_cols.append(roc60_x_atr20)

        if atr14_col in df_out.columns:
            roc60_x_atr14 = _col("roc60_x_atr14")
            df_out[roc60_x_atr14] = df_out[roc60_col] * df_out[atr14_col]
            created_cols.append(roc60_x_atr14)

    # -------------------------
    # 2) Gestation (08:00-09:00): roc_20/roc_30 y mom_5/mom_10
    #    (se crean para todo el día; se evalúan en ventana)
    # -------------------------
    if roc20_col in df_out.columns:
        roc20_minus_roc60 = _col("roc20_minus_roc60")
        df_out[roc20_minus_roc60] = df_out[roc20_col] - df_out[roc60_col]
        created_cols.append(roc20_minus_roc60)


    if (mom5_col in df_out.columns) and (mom10_col in df_out.columns):
        mom5_minus_mom10 = _col("mom5_minus_mom10")
        df_out[mom5_minus_mom10] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(mom5_minus_mom10)

    return df_out, created_cols


## **2.1. Aplicación**

In [19]:
mnq_intraday, interaction_cols = add_interactions(
    mnq_intraday,
    date_col="date",
    ema_col="ema_60",
    roc60_col="roc_60",
    roc20_col="roc_20",
    roc30_col="roc_30",
    mom5_col="mom_5",
    mom10_col="mom_10",
    atr14_col="atr_norm_14",
    atr20_col="atr_norm_20",
)

print("Interacciones creadas:")
print(interaction_cols)


Interacciones creadas:
['roc60_x_atr20', 'roc60_x_atr14', 'roc20_minus_roc60', 'mom5_minus_mom10']


# **4. Remover columnas base sin valor**

Removemos las columnas base que no aportan valor `open`, `high`, `low` y `volume`:

In [20]:
mnq_intraday.drop(columns=["open", "high", "low", "volume"], inplace=True)

In [21]:
mnq_intraday.columns

Index(['date', 'close', 'minute_of_day', 'is_premarket', 'is_opening',
       'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue',
       'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90',
       'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5',
       'roc_20', 'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
       'roc20_minus_roc60', 'mom5_minus_mom10'],
      dtype='object')

In [22]:
info_mnq = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq)

Dataset: mnq_intraday
Shape: (700595, 29)
Columns: ['date', 'close', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14', 'roc20_minus_roc60', 'mom5_minus_mom10']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


Obtenemos la hora de inicio y hora final

In [23]:
# Verificación posterior
start_time_full_day = info_mnq["datetime_min"][11:16]
final_time_full_day = info_mnq["datetime_max"][11:16]

# **6. Creación de datasets independientes por target**

Columnas a conservar por dataset :

In [24]:
mnq_intraday.columns

Index(['date', 'close', 'minute_of_day', 'is_premarket', 'is_opening',
       'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue',
       'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90',
       'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5',
       'roc_20', 'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
       'roc20_minus_roc60', 'mom5_minus_mom10'],
      dtype='object')

In [25]:
delta_60_cols = [
    'date', 'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight',
    'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
    'close',
    'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20',
    'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60', 'mom5_minus_mom10',
    'delta_60',
       ]

delta_90_cols = [
    'date', 'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight',
    'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
    'close',
    'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20',
    'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60', 'mom5_minus_mom10',
    'delta_90',
       ]

ret_60_cols = [
    'date', 'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight',
    'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
    'close',
    'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20',
    'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60', 'mom5_minus_mom10',
    'ret_60',
       ]

ret_90_cols = [
    'date', 'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight',
    'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri',
    'close',
    'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20',
    'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60', 'mom5_minus_mom10',
    'ret_90',
       ]

In [26]:
missing = [c for c in delta_60_cols if c not in mnq_intraday.columns]
if missing:
    raise ValueError(f"Faltan columnas en mnq_intraday: {missing}")

mnq_delta_60 = mnq_intraday[delta_60_cols].copy()

In [27]:
missing = [c for c in delta_90_cols if c not in mnq_intraday.columns]
if missing:
    raise ValueError(f"Faltan columnas en mnq_intraday: {missing}")

mnq_delta_90 = mnq_intraday[delta_90_cols].copy()

In [28]:
missing = [c for c in ret_60_cols if c not in mnq_intraday.columns]
if missing:
    raise ValueError(f"Faltan columnas en mnq_intraday: {missing}")

mnq_ret_60 = mnq_intraday[ret_60_cols].copy()

In [29]:
missing = [c for c in ret_90_cols if c not in mnq_intraday.columns]
if missing:
    raise ValueError(f"Faltan columnas en mnq_intraday: {missing}")

mnq_ret_90 = mnq_intraday[ret_90_cols].copy()

# **7. Activación de feature por ventana**

**Target: `delta_60`**

| delta_60  | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| premarket  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 | 1 | 1 | 1 | 1 |
| opening  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

**Target: `delta_90`**

| delta_90  | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| premarket   | 1 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| opening  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

**Target: `ret_60`**

| ret_60    | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| premarket   | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 1 | 1 |
| opening  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |


**Target: `ret_90`**

| ret_90    | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| premarket   | 1 | 1 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 0 |
| opening  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |



## **7.1. Implementación**

### **Crear columnas para flags**

In [30]:
features_with_flag =   [
    'atr_norm_14', 'atr_norm_20',
    'ema_60',
    'mom_10', 'mom_5',
    'roc_20', 'roc_30', 'roc_60',
    'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60',
    'mom5_minus_mom10']

In [31]:
datasets = [
    mnq_delta_60,
    mnq_delta_90,
    mnq_ret_60,
    mnq_ret_90,
]

for df in datasets:
    for col in features_with_flag:
        if col in df.columns:
            idx = df.columns.get_loc(col)
            df.insert(idx + 1, f"{col}_flag", 0)

### **Definir ventanas**

In [34]:
start_premarket = '08:30'
final_premarket = '09:29'

start_opening = '09:30'
final_opening = '10:29'

In [35]:
print(f'full day: {start_time_full_day} - {final_time_full_day}')
print(f'premarket: {start_premarket} - {final_premarket}')
print(f'opening: {start_opening} - {final_opening}')

full day: 05:30 - 14:30
premarket: 08:30 - 09:29
opening: 09:30 - 10:29


In [37]:
# Helper para convertir HH:MM a minute_of_day
def hhmm_to_minute(hhmm: str) -> int:
    h, m = map(int, hhmm.split(":"))
    return h * 60 + m

windows = {
    "full_day":  (hhmm_to_minute("05:30"), hhmm_to_minute("14:30")),
    "premarket": (hhmm_to_minute("08:30"), hhmm_to_minute("09:29")),
    "opening":   (hhmm_to_minute("09:30"), hhmm_to_minute("10:29")),
}


In [38]:
windows['full_day'][0]

330

### **Planes de activación**

In [39]:
activation_plan_delta_60 = {
    "full_day": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':0,
        'mom_5':0,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "premarket": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':0,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    },

    "opening": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    }
}

In [40]:
activation_plan_delta_90 = {
    "full_day": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':0,
        'mom_5':0,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "premarket": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':0,
        'mom_10':0,
        'mom_5':0,
        'roc_20':0,
        'roc_30':0,
        'roc_60':0,
        'roc60_x_atr20':0,
        'roc60_x_atr14':0,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "opening": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    }
}

In [41]:
activation_plan_ret_60 = {
    "full_day": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':0,
        'mom_5':0,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "premarket": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':0,
        'roc60_x_atr20':0,
        'roc60_x_atr14':0,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    },

    "opening": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    }
}

In [42]:
activation_plan_ret_90 = {
    "full_day": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':0,
        'mom_5':0,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "premarket": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':0,
        'mom_10':0,
        'mom_5':0,
        'roc_20':0,
        'roc_30':0,
        'roc_60':0,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':0,
    },

    "opening": {
        'atr_norm_14':1,
        'atr_norm_20':1,
        'ema_60':1,
        'mom_10':1,
        'mom_5':1,
        'roc_20':1,
        'roc_30':1,
        'roc_60':1,
        'roc60_x_atr20':1,
        'roc60_x_atr14':1,
        'roc20_minus_roc60':1,
        'mom5_minus_mom10':1,
    }
}

### **Función de aplicación de plan**

In [43]:
from typing import Dict, Tuple, List, Optional

def aplicar_plan_activacion(
    df,
    activation_plan: Dict[str, Dict[str, int]],
    windows: Dict[str, Tuple[int, int]],
    *,
    minute_col: str = "minute_of_day",
    priority: Optional[List[str]] = None,
    reset_flags_to_zero: bool = True,
):
    """
    Aplica un plan de activación por ventanas con prioridad.
    - Escribe 0/1 (no solo 1) dentro de cada ventana.
    - Permite priorizar sub-ventanas sobre full_day.

    priority:
      Lista de ventanas en el orden en que deben aplicarse.
      Regla práctica: aplicar primero la más general y al final las más específicas
      si quieres que las específicas "pisen" a las generales.
    """

    # 1) Reset (recomendado): dejar todos los flags en 0 antes de aplicar reglas
    if reset_flags_to_zero:
        flag_cols = [c for c in df.columns if c.endswith("_flag")]
        if flag_cols:
            df.loc[:, flag_cols] = 0

    # 2) Orden de aplicación (para que lo último tenga prioridad y "pise")
    if priority is not None:
        order = [w for w in priority if w in windows]
    else:
        # Default: aplicar primero ventanas grandes y al final las chicas (las chicas pisan)
        order = sorted(windows.keys(), key=lambda k: (windows[k][1] - windows[k][0]), reverse=True)

    # 3) Aplicar plan
    for window_name in order:
        if window_name not in activation_plan:
            continue

        start_min, end_min = windows[window_name]
        mask = (df[minute_col] >= start_min) & (df[minute_col] <= end_min)

        plan = activation_plan[window_name]
        for feature, flag_value in plan.items():
            flag_col = f"{feature}_flag"
            if flag_col in df.columns:
                df.loc[mask, flag_col] = int(flag_value)  # setea 0 o 1

    return df



### **Aplicación de plan**

In [44]:
mnq_delta_60 = aplicar_plan_activacion(
    df=mnq_delta_60,
    activation_plan=activation_plan_delta_60,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # lo último pisa a lo anterior
    reset_flags_to_zero=True,
)

In [45]:
mnq_delta_90 = aplicar_plan_activacion(
    df=mnq_delta_90,
    activation_plan=activation_plan_delta_90,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # lo último pisa a lo anterior
    reset_flags_to_zero=True,
)

In [46]:
mnq_ret_60 = aplicar_plan_activacion(
    df=mnq_ret_60,
    activation_plan=activation_plan_ret_60,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # lo último pisa a lo anterior
    reset_flags_to_zero=True,
)

In [47]:
mnq_ret_90 = aplicar_plan_activacion(
    df=mnq_ret_90,
    activation_plan=activation_plan_ret_90,
    priority=["full_day", "premarket", "opening"],  # lo último pisa a lo anterior
    reset_flags_to_zero=True,
    windows=windows,
)

### **Verificación**

In [48]:
import pandas as pd
from typing import Dict, Tuple, List, Optional

def verificar_flags_en_todas_las_ventanas_con_prioridad(
    df: pd.DataFrame,
    *,
    activation_plan: Dict[str, Dict[str, int]],
    windows: Dict[str, Tuple[int, int]],
    priority: List[str],
    minute_col: str = "minute_of_day",
) -> pd.DataFrame:
    """
    Verifica flags por ventana respetando prioridad:
    - La ventana i se verifica SOLO en filas que no estén cubiertas por ventanas
      con mayor prioridad (las que vienen después en 'priority' si estas "pisan").

    Convención recomendada:
      priority = ["full_day", "premarket", "opening"]
      => opening pisa a premarket pisa a full_day
      => se verifica full_day excluyendo premarket+opening
    """
    results = []
    # Precalcular máscaras por ventana
    masks = {}
    for w, (a, b) in windows.items():
        masks[w] = (df[minute_col] >= a) & (df[minute_col] <= b)

    # Ventanas que pisan: las que están "después" en priority
    for i, w in enumerate(priority):
        if w not in activation_plan or w not in windows:
            continue

        mask_w = masks[w].copy()

        # excluir ventanas de mayor prioridad (las posteriores en la lista)
        for w_higher in priority[i+1:]:
            if w_higher in masks:
                mask_w = mask_w & (~masks[w_higher])

        plan = activation_plan[w]

        for feature, expected in plan.items():
            flag_col = f"{feature}_flag"
            if flag_col not in df.columns:
                results.append({
                    "window": w, "feature": feature, "expected": expected,
                    "ok": None, "fail": None, "fail_rate": None, "status": "MISSING_FLAG"
                })
                continue

            s = df.loc[mask_w, flag_col]
            total = len(s)
            ok = (s == expected).sum()
            fail = (s != expected).sum()

            results.append({
                "window": w,
                "feature": feature,
                "expected": int(expected),
                "ok": int(ok),
                "fail": int(fail),
                "fail_rate": float(fail) / total if total else 0.0,
                "status": "OK" if fail == 0 else "FAIL",
                "n_checked": int(total),
            })

    return pd.DataFrame(results).sort_values(
        ["window", "fail_rate", "feature"],
        ascending=[True, False, True]
    )


In [49]:
verif_delta_60 = verificar_flags_en_todas_las_ventanas_con_prioridad(
    mnq_delta_60,
    activation_plan=activation_plan_delta_60,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # opening pisa a premarket pisa a full_day
)

verif_delta_90 =  verificar_flags_en_todas_las_ventanas_con_prioridad(
    mnq_delta_90,
    activation_plan=activation_plan_delta_90,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # opening pisa a premarket pisa a full_day
)

verif_ret_60 =  verificar_flags_en_todas_las_ventanas_con_prioridad(
    mnq_ret_60,
    activation_plan=activation_plan_ret_60,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # opening pisa a premarket pisa a full_day
)

verif_ret_90 = verificar_flags_en_todas_las_ventanas_con_prioridad(
    mnq_ret_90,
    activation_plan=activation_plan_ret_90,
    windows=windows,
    priority=["full_day", "premarket", "opening"],  # opening pisa a premarket pisa a full_day
)

In [50]:
display(verif_delta_60)
display(verif_delta_90)
display(verif_ret_60)
display(verif_ret_90)

,window,feature,expected,ok,fail,fail_rate,status,n_checked
0,full_day,atr_norm_14,1,545195,0,0.0,OK,545195
1,full_day,atr_norm_20,1,545195,0,0.0,OK,545195
2,full_day,ema_60,1,545195,0,0.0,OK,545195
11,full_day,mom5_minus_mom10,0,545195,0,0.0,OK,545195
3,full_day,mom_10,0,545195,0,0.0,OK,545195
4,full_day,mom_5,0,545195,0,0.0,OK,545195
10,full_day,roc20_minus_roc60,1,545195,0,0.0,OK,545195
9,full_day,roc60_x_atr14,1,545195,0,0.0,OK,545195
8,full_day,roc60_x_atr20,1,545195,0,0.0,OK,545195
5,full_day,roc_20,1,545195,0,0.0,OK,545195


,window,feature,expected,ok,fail,fail_rate,status,n_checked
0,full_day,atr_norm_14,1,545195,0,0.0,OK,545195
1,full_day,atr_norm_20,1,545195,0,0.0,OK,545195
2,full_day,ema_60,1,545195,0,0.0,OK,545195
11,full_day,mom5_minus_mom10,0,545195,0,0.0,OK,545195
3,full_day,mom_10,0,545195,0,0.0,OK,545195
4,full_day,mom_5,0,545195,0,0.0,OK,545195
10,full_day,roc20_minus_roc60,1,545195,0,0.0,OK,545195
9,full_day,roc60_x_atr14,1,545195,0,0.0,OK,545195
8,full_day,roc60_x_atr20,1,545195,0,0.0,OK,545195
5,full_day,roc_20,1,545195,0,0.0,OK,545195


,window,feature,expected,ok,fail,fail_rate,status,n_checked
0,full_day,atr_norm_14,1,545195,0,0.0,OK,545195
1,full_day,atr_norm_20,1,545195,0,0.0,OK,545195
2,full_day,ema_60,1,545195,0,0.0,OK,545195
11,full_day,mom5_minus_mom10,0,545195,0,0.0,OK,545195
3,full_day,mom_10,0,545195,0,0.0,OK,545195
4,full_day,mom_5,0,545195,0,0.0,OK,545195
10,full_day,roc20_minus_roc60,1,545195,0,0.0,OK,545195
9,full_day,roc60_x_atr14,1,545195,0,0.0,OK,545195
8,full_day,roc60_x_atr20,1,545195,0,0.0,OK,545195
5,full_day,roc_20,1,545195,0,0.0,OK,545195


,window,feature,expected,ok,fail,fail_rate,status,n_checked
0,full_day,atr_norm_14,1,545195,0,0.0,OK,545195
1,full_day,atr_norm_20,1,545195,0,0.0,OK,545195
2,full_day,ema_60,1,545195,0,0.0,OK,545195
11,full_day,mom5_minus_mom10,0,545195,0,0.0,OK,545195
3,full_day,mom_10,0,545195,0,0.0,OK,545195
4,full_day,mom_5,0,545195,0,0.0,OK,545195
10,full_day,roc20_minus_roc60,1,545195,0,0.0,OK,545195
9,full_day,roc60_x_atr14,1,545195,0,0.0,OK,545195
8,full_day,roc60_x_atr20,1,545195,0,0.0,OK,545195
5,full_day,roc_20,1,545195,0,0.0,OK,545195


# **8. Guardado de datasets**


In [51]:
OUT_PARQUET_DELTA_60 = Path(os.environ.get("OUT_PARQUET_DELTA_60", "data/features/mnq_delta_60.parquet"))
OUT_PARQUET_DELTA_90 = Path(os.environ.get("OUT_PARQUET_DELTA_90", "data/features/mnq_delta_90.parquet"))
OUT_PARQUET_RET_60 = Path(os.environ.get("OUT_PARQUET_RET_60", "data/features/mnq_ret_60.parquet"))
OUT_PARQUET_RET_90 = Path(os.environ.get("OUT_PARQUET_RET_90", "data/features/mnq_ret_90.parquet"))

OUT_PARQUET_DELTA_60 = DRIVE_DIR / OUT_PARQUET_DELTA_60
OUT_PARQUET_DELTA_90 = DRIVE_DIR / OUT_PARQUET_DELTA_90
OUT_PARQUET_RET_60 = DRIVE_DIR / OUT_PARQUET_RET_60
OUT_PARQUET_RET_90 = DRIVE_DIR / OUT_PARQUET_RET_90

In [52]:
for path, df in [
    (OUT_PARQUET_DELTA_60, mnq_delta_60),
    (OUT_PARQUET_DELTA_90, mnq_delta_90),
    (OUT_PARQUET_RET_60,   mnq_ret_60),
    (OUT_PARQUET_RET_90,   mnq_ret_90),
]:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=True)
    print(f"[OK] Guardado: {path}")

[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_delta_60.parquet
[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_delta_90.parquet
[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_ret_60.parquet
[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_ret_90.parquet
